# 🎵 Análisis Exploratorio — Spotify Music Dataset

## Objetivo
Analizar 4000 canciones de Spotify (rock, pop, rap, reggaeton) para descubrir
qué características tienen las canciones más populares.

## Pasos de este notebook
1. **Carga y revisión** del dataset limpio
2. **Estadísticas descriptivas** por género
3. **Distribuciones** — histogramas de `tempo`, `energy`, `danceability`
4. **Correlaciones** — ¿qué features se relacionan más con `popularity`?
5. **Visualizaciones** — scatter plots y heatmaps
6. **Conclusiones** — qué hace popular una canción

## Dataset
- 4000 canciones balanceadas (1000 por género)
- Fuente: Spotify via Kaggle (`maharshipandya/spotify-tracks-dataset`)
- Features principales: `danceability`, `energy`, `tempo`, `valence`, `popularity`

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Cofigurar el estilo 
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (10,6)

# Cargar DataSet
BASE_DIR = Path().resolve().parent
df = pd.read_csv(BASE_DIR / "data" / "processed" / "spotify_clean.csv")

print(df.shape)
df.head()



(4000, 10)


,artists,track_name,popularity,duration_ms,explicit,danceability,energy,valence,tempo,track_genre
0,Horrorpops,Ghouls,36,126640,False,0.570,0.978,0.762,109.458,rock
1,GAYLE,abcdefu,0,168601,True,0.695,0.540,0.415,121.932,rock
2,Las Pastillas del Abuelo,Hasta Acá Nos Ayudó Dios!,48,378386,False,0.622,0.650,0.638,148.004,rock
3,Soundgarden,Spoonman,0,246920,False,0.258,0.904,0.832,186.054,rock
4,Weezer,Records,4,208920,False,0.658,0.773,0.660,100.006,rock


In [13]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df.duplicated().sum())

(4000, 10)
artists             str
track_name          str
popularity        int64
duration_ms       int64
explicit           bool
danceability    float64
energy          float64
valence         float64
tempo           float64
track_genre         str
dtype: object
artists         0
track_name      0
popularity      0
duration_ms     0
explicit        0
danceability    0
energy          0
valence         0
tempo           0
track_genre     0
dtype: int64
568


In [14]:
df = df.drop_duplicates().reset_index(drop=True)
print(f"Filas después de eliminar duplicados: {df.shape[0]}")

Filas después de eliminar duplicados: 3432


## 1. Estadísticas Descriptivas por Género

Antes de visualizar, necesitamos entender los números crudos.

**¿Qué queremos saber?**
- ¿Qué género tiene canciones más energéticas en promedio?
- ¿Qué género es más "bailable"?
- ¿Hay diferencias claras de tempo entre géneros?

**Método:** agrupar el DataFrame por `track_genre` y calcular
estadísticas agregadas por grupo.

In [15]:
# Agrupar el Dataset por "track_genre" y calcular estadisticas agregadas por grupo
df_stats = df.groupby("track_genre").agg({
    "danceability": "mean",
    "energy": "mean",
    "valence": "mean",
    "tempo": "mean",
    "popularity": "mean"
}).round(3)
print(df_stats)

             danceability  energy  valence    tempo  popularity
track_genre                                                    
pop                 0.571   0.640    0.505  125.506      41.339
rap                 0.721   0.691    0.548  118.336      45.611
reggaeton           0.754   0.726    0.634  120.007      31.045
rock                0.522   0.734    0.499  125.579      38.702


## Conclusiones — Estadísticas Descriptivas

| Género | Popularity media |
|---|---|
| Rap | 45.6 ⭐ |
| Pop | 41.3 |
| Rock | 38.7 |
| Reggaeton | 31.0 |

**Observaciones:**
- **Rap** es el más popular, posiblemente por la presencia de artistas
  mainstream muy escuchados (Drake, Kendrick Lamar, Eminem)
- **Reggaeton** siendo el género más bailable, es el menos popular —
  probable sesgo del dataset hacia el mercado anglosajón
- **Pop** es el segundo más popular a pesar de ser el género más
  comercial a nivel global
- **Rock** ocupa el tercer lugar

> ⚠️ Los datos reflejan el dataset, no necesariamente la realidad global.
> Se debe considerar el sesgo de muestra antes de generalizar conclusiones.

## 2. Distribución de Tempo por Género

El tempo (BPM) nos dice qué tan rápida es una canción.

**Hipótesis:** rock tendrá los BPM más altos, reggaeton los más constantes.

Los gráficos se generan desde `src/visualize.py` y se exportan a `visualizations/`.

## Conclusiones — Distribución de Tempo por Género

- **Rock:** pico principal en ~125 BPM, distribución amplia (50–210 BPM)
- **Pop:** pico principal en ~120 BPM, distribución similar al rock
- **Rap:** concentrado por debajo de 100 BPM, el género más lento de los cuatro
- **Reggaeton:** pico principal en ~90 BPM, con un segundo pico en ~180 BPM
  probablemente causado por detección de tempo a doble velocidad por parte
  de Spotify — error conocido del algoritmo de análisis de audio

> Rock y pop son los géneros más rápidos. Rap y reggaeton son más lentos
> pero con patrones rítmicos más marcados.

## Conclusiones — Energía vs. Danceability

- **Rock (azul):** alta energía pero baja danceability. Se concentra en la
  parte derecha e inferior del gráfico.
- **Rap (verde) y Reggaeton (rojo):** altamente bailables y energéticos.
  Se concentran en la zona superior derecha. Ambos géneros se solapan
  bastante, lo que sugiere que son difíciles de distinguir solo con estas
  dos variables.
- **Pop (naranja):** distribuido por todo el gráfico, sin una zona clara de
  concentración. Como género comercial y cotidiano, sus canciones abarcan
  un rango amplio de características.

> La falta de separación clara entre géneros justifica el uso de K-means
> más adelante para encontrar agrupaciones naturales en los datos.

## Conclusiones — Correlación entre Variables

- Todos los valores de correlación con `popularity` están muy cercanos a 0,
  lo que indica que **ninguna variable de audio tiene relación fuerte con la popularidad**.
- `duration_ms` es la variable con mayor correlación (0.07), pero sigue siendo prácticamente nula.
- La hipótesis inicial de que `valence` sería la más correlacionada no se cumplió.

> La popularidad de una canción no depende principalmente de sus características
> de audio. Depende más de factores externos como el artista, el marketing,
> la época y la plataforma.

> Este hallazgo es relevante: si los géneros no se separan claramente por
> audio features y la popularidad tampoco depende de ellas, el K-means
> nos ayudará a encontrar agrupaciones naturales que los datos escondem.

## Conclusiones — Distribución de Popularidad por Género

- **Rock:** género inconsistente. Rango amplio de popularidad (0–93),
  con la mediana en ~40. Hay bandas muy famosas pero la mayoría
  son poco conocidas.
- **Pop:** el género más estable en la parte baja. Pocas canciones
  con popularidad muy baja, canciones más parecidas entre sí.
- **Rap:** el género más consistentemente popular. Mediana en ~60,
  la más alta de los cuatro. La mayoría de sus canciones son populares.
- **Reggaeton:** el género más desigual. La mediana está cerca de 0
  — la mayoría de canciones son desconocidas — pero hay algunos
  superéxitos que llegan a 97 (Bad Bunny, Daddy Yankee).

> En reggaeton o eres un superéxito o no te conoce nadie.
> En rap casi todos los artistas tienen popularidad similar y alta.

## Conclusiones — K-Means Clustering

| Cluster | Género dominante | Canciones |
|---|---|---|
| Cluster 0 | Pop | 417 |
| Cluster 1 | Reggaeton | 245 |
| Cluster 2 | Rock | 467 |
| Cluster 3 | Reggaeton + Rap | 620 + 484 |

**Hallazgos:**
- K-Means separó bien **rock** y **pop** — tienen características de audio
  suficientemente distintas para formar clusters propios.
- **Rap y reggaeton comparten el Cluster 3** — confirma lo observado en el
  scatter plot: ambos géneros son difíciles de separar por características
  de audio ya que comparten valores similares de `danceability`, `energy`,
  `valence` y `tempo`.

> K-Means no es perfecto para clasificar géneros musicales, pero revela
> agrupaciones naturales en los datos que van más allá de las etiquetas
> de género.